# 02. LangGraph 기반 LEED 버전 표준화

## 연구 핵심: 왜 LangGraph인가?

### 기존 연구의 문제
선행 연구들(Deng et al., 2021; Zuo et al., 2022 등)은 v2.2/v3/v4 데이터를 단순 합산하거나,
버전별 점수를 그대로 비교하여 신뢰성이 낮은 분석 결과를 도출했습니다.

### LangGraph의 역할
```
[Mapper Agent] → [Validator Agent] → 검증 통과? → END
      ↑                  ↓ NO (피드백)
      └──────────────────┘ (순환 루프)
```

- **Mapper Agent**: 구버전 카테고리를 v5로 매핑 (비율 계산 + LLM 추론)
- **Validator Agent**: 건축 환경적 타당성 검증
- **순환 구조**: 검증 실패 시 피드백을 받아 재매핑 → 사람의 편향 배제

In [ ]:
import sys
sys.path.insert(0, '..')

import os
from dotenv import load_dotenv
load_dotenv('../.env')

import pandas as pd
from IPython.display import display, Image

from src.data.loader import LEEDDataLoader
from src.data.preprocessor import LEEDPreprocessor
from src.langgraph_workflow.graph import build_standardization_graph, run_standardization, run_batch_standardization
from src.langgraph_workflow.state import LEEDStandardizationState

print('라이브러리 로딩 완료')
print(f'   OPENAI_API_KEY: {"설정됨" if os.getenv("OPENAI_API_KEY") else "미설정"}')

## 1. LangGraph 워크플로우 시각화

In [ ]:
graph = build_standardization_graph()

# 그래프 구조 출력
try:
    graph_image = graph.get_graph().draw_mermaid_png()
    display(Image(graph_image))
    with open('../outputs/figures/langgraph_workflow.png', 'wb') as f:
        f.write(graph_image)
    print('그래프 이미지 저장 완료')
except Exception as e:
    print(f'그래프 이미지 생성 실패 (필수 아님): {e}')
    print('\n워크플로우 구조:')
    print('START → [mapper] → [validator] → (조건부) → [finalize] → END')
    print('                       ↑              ↓ (재시도)')
    print('                       └──────────────┘')

## 2. 데이터 로딩

In [ ]:
loader = LEEDDataLoader(data_dir='../data/raw')

# ─────────────────────────────────────────────────────────────────────
# 실제 데이터가 있을 경우:
# df = pd.read_csv('../data/processed/korea_leed_raw.csv')
# ─────────────────────────────────────────────────────────────────────
# 샘플 데이터 사용
df = loader.create_sample_data()

print(f'데이터 로딩 완료: {len(df)}개 프로젝트')
print(f'버전 분포:\n{df["version"].value_counts()}')

## 3. 단일 프로젝트 테스트 실행

LangGraph 워크플로우의 실제 동작을 한 건으로 먼저 확인합니다.

In [ ]:
# v3 프로젝트 1개 선택 (버전 간 차이가 큰 케이스)
v3_sample = df[df['version'] == 'v3'].iloc[0].to_dict()

# 프로젝트 딕셔너리 구성
# ─────────────────────────────────────────────────────────────────────
# 실제 데이터를 사용할 경우, 이 딕셔너리의 키를 실제 컬럼명에 맞게 수정하세요
# ─────────────────────────────────────────────────────────────────────
project_data = {
    'project_id': v3_sample.get('project_id', 'TEST-001'),
    'project_name': v3_sample.get('project_name', '테스트 프로젝트'),
    'version': v3_sample['version'],
    'building_type': v3_sample.get('building_type', 'Office'),
    'gross_area_sqm': v3_sample.get('gross_area_sqm', 30000),
    'certification_level': v3_sample.get('certification_level', 'Gold'),
    'categories': {
        cat: v3_sample.get(f'score_{cat}', 0)
        for cat in ['SS', 'WE', 'EA', 'MR', 'IEQ', 'IN', 'RP']
        if f'score_{cat}' in v3_sample
    },
    'total_score_raw': v3_sample.get('total_score_raw', 0),
}

print('테스트 프로젝트:')
for k, v in project_data.items():
    print(f'  {k}: {v}')

In [ ]:
# API Key가 설정된 경우에만 LangGraph 실행
import os

if os.getenv('OPENAI_API_KEY'):
    print('LangGraph 워크플로우 실행...')
    result = run_standardization(project_data, max_iterations=3)

    print('\n=== 워크플로우 실행 결과 ===')
    print(f'상태: {result["status"]}')
    print(f'반복 횟수: {result["current_iteration"]}')
    print(f'\n로그:')
    for log in result['logs']:
        print(f'  {log}')

    if result.get('final_v5_data'):
        print(f'\n=== v5 표준화 결과 ===')
        for k, v in result['final_v5_data'].items():
            print(f'  {k}: {v}')
else:
    print('[주의] OPENAI_API_KEY가 설정되지 않아 수학적 비율 계산 방식으로 폴백합니다.')
    # 수학적 비율 계산 방식 (API 없이도 동작)
    preprocessor = LEEDPreprocessor()
    test_df = pd.DataFrame([v3_sample])
    normalized = preprocessor.normalize_by_version(test_df)
    print('수학적 비율 계산 결과:')
    v5_cols = [c for c in normalized.columns if c.startswith('score_v5_')]
    print(normalized[v5_cols].to_string())

## 4. 전체 데이터셋 일괄 표준화

### 두 가지 방식:
1. **LangGraph 방식** (API Key 필요): LLM + Validator 순환 검증
2. **수학적 비율 계산** (API 불필요): 빠른 처리, 기본 표준화

In [ ]:
preprocessor = LEEDPreprocessor()

# ─────────────────────────────────────────────────────────────────────
# 방법 1: 수학적 비율 계산 (API 불필요, 빠름) - 기본 사용
# ─────────────────────────────────────────────────────────────────────
print('방법 1: 수학적 비율 계산으로 전체 표준화...')
standardized_df = preprocessor.run_pipeline(df)

v5_score_cols = [c for c in standardized_df.columns if c.startswith('score_v5_')]
print(f'\nv5 환산 점수 컬럼: {v5_score_cols}')
standardized_df[['version', 'certification_level', 'total_score_v5'] + v5_score_cols].head(10)

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# 방법 2: LangGraph 방식 (API Key 있을 때, 소규모 테스트 권장)
# 전체 451개에 실행하면 API 비용 발생!
# 테스트는 10개만 실행 권장
# ─────────────────────────────────────────────────────────────────────

if os.getenv('OPENAI_API_KEY'):
    print('방법 2: LangGraph 방식으로 10개 테스트...')
    test_projects = []
    for _, row in df.head(10).iterrows():
        project = {
            'project_id': row.get('project_id', 'UNKNOWN'),
            'project_name': row.get('project_name', ''),
            'version': row['version'],
            'building_type': row.get('building_type', ''),
            'gross_area_sqm': row.get('gross_area_sqm', 0),
            'certification_level': row.get('certification_level', ''),
            'categories': {
                cat: row.get(f'score_{cat}', 0)
                for cat in ['SS', 'WE', 'EA', 'MR', 'IEQ', 'IN', 'RP', 'LT', 'IP']
                if f'score_{cat}' in row
            },
            'total_score_raw': row.get('total_score_raw', 0),
        }
        test_projects.append(project)

    langgraph_results = run_batch_standardization(test_projects, max_iterations=3)
    langgraph_df = pd.DataFrame(langgraph_results)
    print(f'\nLangGraph 표준화 결과 ({len(langgraph_df)}개):')
    display(langgraph_df.head())
else:
    print('[주의] OPENAI_API_KEY 미설정 - 방법 1 (수학적 계산) 결과를 사용합니다.')

## 5. 표준화 결과 검증 및 저장

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

# 표준화 전후 총점 분포 비교
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 원본 총점 분포
if 'total_score_raw' in standardized_df.columns:
    standardized_df['total_score_raw'].hist(
        bins=30, ax=axes[0], color='#4ECDC4', edgecolor='white'
    )
    axes[0].set_title('원본 총점 분포\n(버전별로 기준 상이)', fontsize=12)
    axes[0].set_xlabel('총점')
    axes[0].set_ylabel('건물 수')

# v5 환산 총점 분포
standardized_df['total_score_v5'].hist(
    bins=30, ax=axes[1], color='#FF6B6B', edgecolor='white'
)
# 등급 구간 표시
for score, label, color in [(40, 'Certified', 'gray'), (50, 'Silver', 'silver'),
                             (60, 'Gold', 'gold'), (80, 'Platinum', 'lightblue')]:
    axes[1].axvline(x=score, color=color, linestyle='--', alpha=0.7, label=f'{label}({score})')
axes[1].set_title('v5 기준 환산 총점 분포\n(LangGraph 표준화 결과)', fontsize=12)
axes[1].set_xlabel('v5 환산 총점 (/110)')
axes[1].set_ylabel('건물 수')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('../outputs/figures/02_standardization_result.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 표준화된 데이터 저장
save_path = '../data/processed/korea_leed_standardized_v5.csv'
standardized_df.to_csv(save_path, index=False, encoding='utf-8-sig')
print(f'표준화 데이터 저장 완료: {save_path}')
print(f'   컬럼 수: {len(standardized_df.columns)}')
print(f'   샘플 수: {len(standardized_df)}')